In [0]:
denials = dbutils.widgets.get("denials")
yrinvo = dbutils.widgets.get("yrinvo")
hchbofficemapping = dbutils.widgets.get("hchbofficemapping")
oblisthistory = dbutils.widgets.get("oblisthistory")
billing_reference = dbutils.widgets.get("billing_reference")
office = dbutils.widgets.get("office")
payerdimension = dbutils.widgets.get("payerdimension")
cubeserviceofficetxnsourcesystem = dbutils.widgets.get("cubeserviceofficetxnsourcesystem")
date = dbutils.widgets.get("date")
client_episode_fs = dbutils.widgets.get("client_episode_fs")
client_episodes_all = dbutils.widgets.get("client_episodes_all")
client = dbutils.widgets.get("client")
denialcodemapping = dbutils.widgets.get("denialcodemapping")
denialtype = dbutils.widgets.get("denialtype")
abilityremittancedetails=dbutils.widgets.get("abilityremittancedetails")
mart_denial=dbutils.widgets.get("mart_denial")

In [0]:
count = spark.sql(f"SELECT COUNT(*) as cnt FROM {denials}").collect()[0]['cnt']
if count == 0 :
    print(f"Full load for {denials} executed")
    spark.sql(f"""
    

INSERT INTO {denials}
(
    reporting_week_ending_date_key,
    denial_date_key,
    source_system_key,
    office_key,
    payor_key,
    client_key,
    enterprise_category_key,
    denial_type_key,
    remark_code,
    patientctlno,
    invoice_number,
    denied_amount,
    loaded_ts
)
SELECT
    CAST(Reporting_Week_Ending_Date_Key AS INT),
    CAST(Denial_Date_Key AS INT),
    CAST(Source_System_Key AS INT),
    CAST(Office_Key AS INT),
    CAST(Payor_Key AS INT),
    CAST(Client_Key AS INT),
    CAST(Enterprise_Category_Key AS INT),
    CAST(Denial_Type_Key AS INT),
    CAST(Remark_Code AS STRING),
    CAST(PatientCtlNo AS STRING),
    CAST(Invoice_Number AS STRING),
    CAST(Denied_Amount AS DOUBLE),
    current_timestamp() AS loaded_ts
FROM {mart_denial}
    """)
 

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW tmp1_base AS
SELECT 
    BranchID,
    InvNum,
    GroupID,
    CltId,
    PrimID
FROM {yrinvo}
WHERE BranchID <> 'COR'
""")


In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW tmp1 AS
SELECT 
    CASE 
        WHEN t.BranchID RLIKE '[A-Z]' THEN om.TargetOfficeNumber
        ELSE t.BranchID
    END AS BranchID,
    t.InvNum,
    t.GroupID,
    t.CltId,
    t.PrimID
FROM tmp1_base t
LEFT JOIN {hchbofficemapping} om
    ON t.BranchID = om.SourceOfficeCode
""")


In [0]:

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW tmp2 AS
SELECT office, invno, payorname, PAYORTYPE, BILLTO, CLIENTNO
FROM (
    SELECT
        office,
        invno,
        payorname,
        PAYORTYPE,
        BILLTO,
        CLIENTNO,
        ROW_NUMBER() OVER (PARTITION BY invno, office ORDER BY invno, office) AS rnb
    FROM {oblisthistory}
    WHERE invno <> 'adv'
) a
WHERE rnb = 1
""")


In [0]:
spark.sql(
    f""" 
INSERT INTO {denials} (
    reporting_week_ending_date_key,
    denial_date_key,
    source_system_key,
    office_key,
    payor_key,
    client_key,
    enterprise_category_key,
    denial_type_key,
    remark_code,
    patientctlno,
    invoice_number,
    denied_amount
)

-- First SELECT - HCHB Denials
SELECT 
    TRY_CAST(REPLACE(dt1.WeekEndingDate, '-', '') AS INT) AS reporting_week_ending_date_key,
    dt1.DateKey AS denial_date_key,
    s.SourceSystemKey AS source_system_key,
    ofc.OfficeKey AS office_key,
    pd.PayerKey AS payor_key,
    b.ClientKey AS client_key,
    dcm.Denial_Code_Key AS enterprise_category_key,
    dl.Denial_Type_Key AS denial_type_key,
    ard.RemitRemarks AS remark_code,
    ard.PatientCtlNo AS patientctlno,
    ard.PatientCtlNo AS invoice_number,
    CAST(ard.TotalServAdjustments AS DECIMAL(19,4)) AS denied_amount
FROM {abilityremittancedetails} ard

JOIN tmp1 inv
    ON TRY_CAST(ard.PatientCtlNo AS STRING) = TRY_CAST(inv.InvNum AS STRING)
    AND (ard.PatientCtlNo LIKE '2[89][90123]%' OR ard.PatientCtlNo LIKE '1%')

LEFT JOIN {office} ofc
    ON ofc.OfficeNumber = ABS(TRY_CAST(inv.BranchID AS INT))

LEFT JOIN {payerdimension} pd
    ON TRY_CAST(inv.GroupID AS STRING) = TRY_CAST(pd.PayerID AS STRING)

LEFT JOIN {cubeserviceofficetxnsourcesystem} s
    ON s.SourceSystemName = 'HCHB'

LEFT JOIN {date} dt1
    ON dt1.CalendarDate = TRY_CAST(ard.CreateDate AS DATE)

LEFT JOIN {client_episode_fs} cefs
    ON TRY_CAST(inv.PrimID AS BIGINT) = TRY_CAST(cefs.cefs_id AS BIGINT)

LEFT JOIN {client_episodes_all} cea
    ON cea.epi_id = cefs.cefs_epiid

LEFT JOIN (
    SELECT * FROM (
        SELECT 
            clientKey,
            SourceSystemId,
            OfficeNumber,
            ROW_NUMBER() OVER (
                PARTITION BY SourceSystemId, OfficeNumber 
                ORDER BY SourceSystemId, OfficeNumber
            ) AS rnb
        FROM {client}
        WHERE OfficeNumber <> 0 
          AND SourceSystem = 'HCHB'
    ) a
    WHERE rnb = 1
) b
    ON TRY_CAST(b.SourceSystemId AS STRING) = TRY_CAST(cea.epi_id AS STRING)
   AND ABS(TRY_CAST(inv.BranchID AS INT)) = b.OfficeNumber

LEFT JOIN {denialcodemapping} dcm
    ON dcm.Code = CASE 
        WHEN LOCATE(')', ard.AdjustmentCodes) = 6 THEN 
            SUBSTRING(REPLACE(ard.AdjustmentCodes, ':', ''), 2, 3)
        WHEN LOCATE(')', ard.AdjustmentCodes) = 7 THEN 
            SUBSTRING(REPLACE(ard.AdjustmentCodes, ':', ''), 2, 4)
        WHEN LOCATE(')', ard.AdjustmentCodes) = 8 THEN 
            SUBSTRING(REPLACE(ard.AdjustmentCodes, ':', ''), 2, 5)
    END

JOIN {denialtype} dl
    ON dl.Denial_Type = CASE 
        WHEN ard.TotalServAdjustments > 0 THEN 'Adjustment'
        WHEN ard.TotalServAdjustments < 0 THEN 'Reversal'
    END

WHERE ard.TotalServAdjustments <> 0 
  AND TRY_CAST(ard.loaddate AS DATE) = CURRENT_DATE()
  AND ard.Grouping = 'BHHC'

UNION ALL

-- Second SELECT - CUBHUB Denials
SELECT 
    TRY_CAST(REPLACE(dt1.WeekEndingDate, '-', '') AS INT) AS reporting_week_ending_date_key,
    dt1.DateKey AS denial_date_key,
    s.SourceSystemKey AS source_system_key,
    ofc.OfficeKey AS office_key,
    pd.PayerKey AS payor_key,
    b.ClientKey AS client_key,
    dcm.Denial_Code_Key AS enterprise_category_key,
    dl.Denial_Type_Key AS denial_type_key,
    ard.RemitRemarks AS remark_code,
    ard.PatientCtlNo AS patientctlno,
    ard.PatientCtlNo AS invoice_number,
    CAST(ard.TotalServAdjustments AS DECIMAL(19,4)) AS denied_amount
FROM {abilityremittancedetails} ard

LEFT JOIN (
    SELECT * FROM (
        SELECT 
            *,
            ROW_NUMBER() OVER (PARTITION BY Name ORDER BY PayerKey) AS rnb
        FROM {payerdimension}
        WHERE SourceSystemKey = 19
    ) a
    WHERE rnb = 1
) pd
    ON ard.PayerName = pd.Name

LEFT JOIN {cubeserviceofficetxnsourcesystem} s
    ON s.SourceSystemName = 'CUBHUB'

LEFT JOIN {date} dt1
    ON dt1.CalendarDate = TRY_CAST(ard.CreateDate AS DATE)

LEFT JOIN (
    SELECT * FROM (
        SELECT 
            clientKey,
            ConformedLastName,
            ConformedFirstName,
            OfficeNumber,
            ROW_NUMBER() OVER (
                PARTITION BY ConformedLastName, ConformedFirstName 
                ORDER BY ClientKey DESC
            ) AS rnb
        FROM {client}
        WHERE OfficeNumber <> 0 
          AND SourceSystem = 'CUBHUB'
    ) a
    WHERE rnb = 1
) b
    ON ard.PatientName = CONCAT(b.ConformedLastName, ', ', b.ConformedFirstName)

LEFT JOIN {office} ofc
    ON ofc.OfficeNumber = b.OfficeNumber

LEFT JOIN {denialcodemapping} dcm
    ON dcm.Code = CASE 
        WHEN LOCATE(')', ard.AdjustmentCodes) = 6 THEN 
            SUBSTRING(REPLACE(ard.AdjustmentCodes, ':', ''), 2, 3)
        WHEN LOCATE(')', ard.AdjustmentCodes) = 7 THEN 
            SUBSTRING(REPLACE(ard.AdjustmentCodes, ':', ''), 2, 4)
        WHEN LOCATE(')', ard.AdjustmentCodes) = 8 THEN 
            SUBSTRING(REPLACE(ard.AdjustmentCodes, ':', ''), 2, 5)
    END

JOIN {denialtype} dl
    ON dl.Denial_Type = CASE 
        WHEN ard.TotalServAdjustments > 0 THEN 'Adjustment'
        WHEN ard.TotalServAdjustments < 0 THEN 'Reversal'
    END

WHERE ard.TotalServAdjustments <> 0 
  AND TRY_CAST(ard.loaddate AS DATE) = CURRENT_DATE()
  AND ard.Grouping = 'BHHC'
  AND (
      ard.PatientCtlNo RLIKE '^[0-9].*[0-9][E-F][a-z][0-9].*[0-9]$'
      OR ard.PatientCtlNo RLIKE '^[0-9].*[0-9][E-F][a-z]S[0-9].*[0-9]$'
      OR ard.PatientCtlNo RLIKE '^M-[0-9].*[0-9][E-F][a-z][0-9].*[0-9]$'
      OR ard.PatientCtlNo RLIKE '^M-[0-9].*[0-9][E-F][a-z]S[0-9].*[0-9]$'
  )

UNION ALL

-- Third SELECT - BEARS Denials
SELECT 
    TRY_CAST(REPLACE(dt1.WeekEndingDate, '-', '') AS INT) AS reporting_week_ending_date_key,
    dt1.DateKey AS denial_date_key,
    CASE WHEN tmp2.INVNO IS NOT NULL THEN s.SourceSystemKey END AS source_system_key,
    ofc.OfficeKey AS office_key,
    pd.PayerKey AS payor_key,
    d.ClientKey AS client_key,
    dcm.Denial_Code_Key AS enterprise_category_key,
    dl.Denial_Type_Key AS denial_type_key,
    b.Remark_Code AS remark_code,
    b.PatientCtlNo AS patientctlno,
    CASE WHEN tmp2.INVNO IS NOT NULL THEN RIGHT(b.PatientCtlNo, 8) END AS invoice_number,
    CAST(b.TotalServAdjustments AS DECIMAL(19,4)) AS denied_amount
FROM (
    SELECT 
        ard.CreateDate,
        ard.Filename,
        ard.PatientCtlNo,
        ard.AdjustmentCodes AS Reason_Code,
        ard.RemitRemarks AS Remark_Code,
        ard.TotalServAdjustments
    FROM {abilityremittancedetails} ard
    
    LEFT JOIN tmp1 inv
        ON TRY_CAST(ard.PatientCtlNo AS STRING) = TRY_CAST(inv.InvNum AS STRING)
        AND (ard.PatientCtlNo LIKE '2[89][90123]%' OR ard.PatientCtlNo LIKE '1%')
    
    WHERE ard.TotalServAdjustments <> 0
      AND inv.InvNum IS NULL
     AND TRY_CAST(ard.loaddate AS DATE) = CURRENT_DATE()
      AND ard.Grouping = 'BHHC'
      AND ard.PatientCtlNo NOT IN (
          SELECT DISTINCT PatientCtlNo 
          FROM {abilityremittancedetails}
          WHERE 
             (
                PatientCtlNo RLIKE '^[0-9].*[0-9]E[a-z][0-9].*[0-9]$'
                OR PatientCtlNo RLIKE '^[0-9].*[0-9]E[a-z]S[0-9].*[0-9]$'
                OR PatientCtlNo RLIKE '^M-[0-9].*[0-9][E-F][a-z][0-9].*[0-9]$'
                OR PatientCtlNo RLIKE '^M-[0-9].*[0-9][E-F][a-z]S[0-9].*[0-9]$'
            )
             AND TRY_CAST(loaddate AS DATE) = CURRENT_DATE()
      )
) b

LEFT JOIN tmp2
    ON tmp2.INVNO = RIGHT(b.PatientCtlNo, 8)
    AND LENGTH(b.PatientCtlNo) <> 7

LEFT JOIN {office} ofc
    ON ofc.OfficeNumber = tmp2.office

LEFT JOIN {payerdimension} pd
    ON pd.PayerID = tmp2.BILLTO

LEFT JOIN {cubeserviceofficetxnsourcesystem} s
    ON s.SourceSystemName = 'BEARS'

LEFT JOIN {date} dt1
    ON dt1.CalendarDate = TRY_CAST(b.CreateDate AS DATE)

LEFT JOIN (
    SELECT * FROM (
        SELECT 
            sourcesystemid,
            officenumber,
            ClientKey,
            ROW_NUMBER() OVER (PARTITION BY sourcesystemid ORDER BY clientkey DESC) AS rnb
        FROM {client}
        WHERE officenumber <> 0 
          AND sourcesystem = 'BEARS'
    ) s
    WHERE rnb = 1
) d
    ON d.SourceSystemId = tmp2.CLIENTNO 
   AND d.OfficeNumber = tmp2.Office

LEFT OUTER JOIN {denialcodemapping} dcm
    ON dcm.Code = CASE 
        WHEN LOCATE(')', b.Reason_Code) = 6 THEN 
            SUBSTRING(REPLACE(b.Reason_Code, ':', ''), 2, 3)
        WHEN LOCATE(')', b.Reason_Code) = 7 THEN 
            SUBSTRING(REPLACE(b.Reason_Code, ':', ''), 2, 4)
        WHEN LOCATE(')', b.Reason_Code) = 8 THEN 
            SUBSTRING(REPLACE(b.Reason_Code, ':', ''), 2, 5)
    END

JOIN {denialtype} dl
    ON dl.Denial_Type = CASE 
        WHEN b.TotalServAdjustments > 0 THEN 'Adjustment'
        WHEN b.TotalServAdjustments < 0 THEN 'Reversal'
    END
    """
)